In [0]:
df= spark.read.csv("/Volumes/data/orders/files/archive/sales.csv",header=True,inferSchema=True)
import pyspark.sql.functions as f


read sample data and check schema


In [0]:
df.printSchema()
display(df.limit(10))

check duplicates

In [0]:
dup = df.groupBy("order_id").count().filter("count > 1").count()
display(dup)

check nulls 

In [0]:
null_count = df.select([
    f.count(f.when(f.col(c).isNull(),1)).alias(c)
    for c in df.columns ])
display(null_count)

check data is valid or not

In [0]:
invalid = df.where((f.col('quantity') <= 0) | (f.col("revenue") <= 0) | (f.col("cost") <= 0))
print(f"invalid count {valid.count()}")

check value correctness

In [0]:
# for revenue
check_revenue = df.withColumn('cal_revenue',f.col('quantity')*(f.col('unit_price')-f.col('discount')))\
    .where(f.col('revenue') != f.col('cal_revenue'))
print(f"wrong revenue count {check_revenue.count()}")

#for profit
check_profit = df.withColumn('cal_profit',f.col('revenue')-f.col('cost'))\
    .where(f.col('profit') != f.col('cal_profit'))
print(f"wrong profit count {check_profit.count()}")

display wrong values  to check difference

In [0]:
wrong_rev = check_revenue.select("order_id","quantity","unit_price","discount","revenue","cal_revenue").limit(10)
display(wrong_rev)

wrong_pro = check_profit.select("order_id","revenue","cost","profit","cal_profit").limit(10)
display(wrong_pro)

first fill revenue with correct value

In [0]:
final_df = check_revenue.withColumn('new_revenue',f.round('cal_revenue',2))\
    .withColumn('New_profit',f.round(f.col('cal_revenue')-f.col('cost'),2))\
    .withColumnRenamed("profit","old_profit")\
    .withColumnRenamed("revenue","old_revenue")\
    .drop("cal_revenue")
display(final_df.limit(10))

write data in delta format

In [0]:
final_df.write.mode('overwrite').format('delta').save('/Volumes/data/orders/files/chocolate/sales')